# Notebook 04: 学习到的 Score Field 可视化

**目标**：把训好的 noise predictor 转成 score field，在 2D 平面上画箭头图，直观看到"score 指向高密度方向"。

**前置**：nb01（需要训好一个 2D DDPM）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 重用 nb01 中的设置
from sklearn.datasets import make_moons
x_train, _ = make_moons(5000, noise=0.05)
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)

T = 1000
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1.0 - betas
alphas_cumprod = alphas.cumprod(0)
sqrt_a = alphas_cumprod.sqrt()
sqrt_1ma = (1 - alphas_cumprod).sqrt()

In [ ]:
class TimeEmb(nn.Module):
    def __init__(self, dim=64):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
        a = t[:, None].float() * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=-1)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.t = TimeEmb(64)
        self.net = nn.Sequential(
            nn.Linear(66, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 2),
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, self.t(t)], dim=-1))

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for step in range(2000):
    idx = torch.randint(0, len(x_train), (512,))
    x0 = x_train[idx]
    t = torch.randint(0, T, (512,), device=device)
    eps = torch.randn_like(x0)
    xt = sqrt_a[t].unsqueeze(-1) * x0 + sqrt_1ma[t].unsqueeze(-1) * eps
    loss = F.mse_loss(model(xt, t), eps)
    opt.zero_grad(); loss.backward(); opt.step()
print(f'Final loss: {loss.item():.4f}')

## 在不同时间步可视化 score field

Score 与 noise 关系：∇_x log p_t(x) = -ε / √(1-ᾱ_t)

在网格点上计算 score，用 quiver 画箭头。

In [ ]:
@torch.no_grad()
def compute_score_field(t_val, grid_n=20):
    xs = torch.linspace(-2, 2, grid_n)
    ys = torch.linspace(-2, 2, grid_n)
    X, Y = torch.meshgrid(xs, ys, indexing='xy')
    pts = torch.stack([X.flatten(), Y.flatten()], dim=-1).to(device)
    t = torch.full((pts.shape[0],), t_val, device=device, dtype=torch.long)
    eps_pred = model(pts, t)
    score = -eps_pred / sqrt_1ma[t_val]
    return X.cpu(), Y.cpu(), score.cpu()

ts_show = [50, 200, 500, 800]
fig, axes = plt.subplots(1, len(ts_show), figsize=(4*len(ts_show), 4))
model.eval()
for ax, t_val in zip(axes, ts_show):
    X, Y, S = compute_score_field(t_val, grid_n=20)
    # 归一化箭头长度以便可视化
    norm = S.norm(dim=-1, keepdim=True).clamp(min=1e-4)
    S_normed = S / norm * (norm / norm.max()).pow(0.5)
    ax.quiver(X, Y, S_normed[:, 0].reshape_as(X), S_normed[:, 1].reshape_as(X),
              alpha=0.7, scale=20)
    # 把训练数据加噪到 t_val 并画出
    if t_val > 0:
        with torch.no_grad():
            xt_train = sqrt_a[t_val] * x_train[:1000] + sqrt_1ma[t_val] * torch.randn_like(x_train[:1000])
        ax.scatter(xt_train[:,0].cpu(), xt_train[:,1].cpu(), s=2, c='red', alpha=0.3)
    else:
        ax.scatter(x_train[:1000,0].cpu(), x_train[:1000,1].cpu(), s=2, c='red', alpha=0.3)
    ax.set_title(f't={t_val}'); ax.set_aspect('equal')
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
plt.tight_layout(); plt.show()

## 观察

- **小 t**：score 在数据附近形成尖锐的"吸引"方向
- **大 t**：数据接近高斯，score 整体指向原点（因为 ∇log N(0,I) = -x）

这正是 Score SDE / Anderson 公式所说的——反向 SDE 沿着 score 走，所以从噪声出发能到达数据分布。

## 思考题

1. 在 t=999 时，理论上 score(x) ≈ -x（标准高斯的 score）。验证一下
2. 在数据点稀疏的区域（如两月之间的间隙），score 指向哪个方向？
3. 把训练步数减少到 200 步，重新跑，观察 score field 的退化